# ReID — Held-out evaluation (1× GPU, `.reidraw` targets)

Evaluates **`model_best.pth`** from the team Hugging Face bucket on the Kaggle unseen-target bundle
[`a7medlabib/reid-unseen-evaluation-targets-v1`](https://www.kaggle.com/datasets/a7medlabib/reid-unseen-evaluation-targets-v1).

**Uses (5 public benchmarks):** Market1501, MSMT17, MARS, DukeMTMC, Occluded-Duke `.reidraw` files.  
**Skips:** `TeamRecorded` (not in paper macro).

**Outputs per dataset:** Rank-1, Rank-5, mAP, CMC ranks 1–20 (%).  
**Macro (public):** unweighted mean over the five benchmarks (paper Table convention).

**Attach on Kaggle:**
- Input: `a7medlabib/reid-unseen-evaluation-targets-v1`
- Secrets: `GITHUB_TOKEN`, `HF_TOKEN`
- GPU: 1× T4 (or P100) is enough

Code is checked out from Git branch **`test`** (eval-only); training notebooks stay pinned on **`main`** until merge.

In [ ]:
# ---- edit these ----
CONDITION = "B6"
SEED = 0
RUN_NAME = ""  # default: f"{CONDITION}_seed{SEED}"
RESEARCH_BRANCH = "test"  # eval-only branch; merge to main after training
HF_BUCKET_ID = "reid-research-hub/reid-loss-study-live"
HF_PREFIX = f"v3_50epoch_28c68a1/{CONDITION}/seed{SEED}"
REPO_URL = "https://github.com/BASSAT-BASSAT/Person-Re-ID-Foundational-model.git"
# --------------------

import os
import subprocess
import sys
from pathlib import Path

import torch
from kaggle_secrets import UserSecretsClient

ROOT = Path("/kaggle/working/Person-Re-ID-Foundational-model-eval")
WORK = Path(f"/kaggle/working/paper_eval/{CONDITION}/seed{SEED}")
WORK.mkdir(parents=True, exist_ok=True)

TARGET_CANDIDATES = [
    Path("/kaggle/input/reid-unseen-evaluation-targets-v1"),
    Path("/kaggle/input/datasets/a7medlabib/reid-unseen-evaluation-targets-v1"),
]
TARGET_ROOT = next((p for p in TARGET_CANDIDATES if (p / "reid_targets_index.json").is_file()), None)
assert TARGET_ROOT is not None, "Attach dataset a7medlabib/reid-unseen-evaluation-targets-v1"

assert torch.cuda.is_available(), "Enable one GPU"
secret_client = UserSecretsClient()
GITHUB_TOKEN = secret_client.get_secret("GITHUB_TOKEN")
HF_TOKEN = secret_client.get_secret("HF_TOKEN")
assert GITHUB_TOKEN and HF_TOKEN, "Enable GITHUB_TOKEN and HF_TOKEN in Kaggle Secrets"
print(f"[PASS] GPU | targets {TARGET_ROOT.name} | {CONDITION} seed {SEED}")

In [ ]:
if not ROOT.is_dir():
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    subprocess.check_call(["git", "clone", "--quiet", clone_url, str(ROOT)])
subprocess.check_call(["git", "fetch", "--quiet", "origin", RESEARCH_BRANCH], cwd=ROOT)
subprocess.check_call(
    ["git", "checkout", "--quiet", "-B", RESEARCH_BRANCH, f"origin/{RESEARCH_BRANCH}"],
    cwd=ROOT,
)
sys.path.insert(0, str(ROOT))
rev = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
subprocess.check_call([sys.executable, str(ROOT / "scripts/setup_fastreid.py"), "--cython"], cwd=ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]", "huggingface_hub>=1.7.1,<2"], cwd=ROOT)
assert (ROOT / "scripts/evaluate_unseen_reidraw.py").is_file(), f"Branch {RESEARCH_BRANCH} missing eval scripts"
print(f"[PASS] Repo origin/{RESEARCH_BRANCH} @ {rev[:12]}…")

In [ ]:
from huggingface_hub import download_bucket_files

ckpt = WORK / "model_best.pth"
remote_ckpt = f"{HF_PREFIX}/model_best.pth"
download_bucket_files(HF_BUCKET_ID, files=[(remote_ckpt, ckpt)], token=HF_TOKEN)
assert ckpt.is_file() and ckpt.stat().st_size > 1024**2, "model_best download failed or too small"
print(f"[PASS] Downloaded {remote_ckpt}")

In [ ]:
name = RUN_NAME or f"{CONDITION}_seed{SEED}"
cmd = [
    sys.executable,
    str(ROOT / "scripts/evaluate_unseen_reidraw.py"),
    "--checkpoint", str(ckpt),
    "--condition", CONDITION,
    "--seed", str(SEED),
    "--target-root", str(TARGET_ROOT),
    "--name", name,
    "--out", str(WORK / "results"),
    "--device", "cuda",
    "--batch-size", "128",
    "--workers", "4",
]
subprocess.check_call(cmd, cwd=ROOT)
print(f"[PASS] Results in {WORK / 'results'}")

In [ ]:
import json

report = json.loads((WORK / "results" / f"{name}.json").read_text(encoding="utf-8"))
print((WORK / "results" / f"{name}.md").read_text(encoding="utf-8"))
macro = report["paper"]["macro_public"]
print("\nMacro (5 public): "
      f"R1={macro['Rank-1']:.2f}%  R5={macro['Rank-5']:.2f}%  mAP={macro['mAP']:.2f}%")